# Train YOLO on Merged Dataset (IDD + RoadDefects)
This notebook trains a YOLO segmentation model on the newly created `Merged_YOLO_Dataset`.

**Prerequisites:**
1. Upload the `Merged_YOLO_Dataset` directory from your local machine to your Google Drive at `/content/drive/MyDrive/SIH_Epilgoue/Dataset/Merged_YOLO_Dataset`.
2. Ensure your Colab Runtime is set to GPU (T4).

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install Dependencies & Check GPU
!pip install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# 2.5 Preprocess Labels: Convert Bounding Boxes to Polygons
import os

def bbox_to_polygon(line_parts):
    class_id = line_parts[0]
    xc = float(line_parts[1])
    yc = float(line_parts[2])
    w = float(line_parts[3])
    h = float(line_parts[4])

    # Calculate corners (top-left, top-right, bottom-right, bottom-left)
    x1 = max(0.0, xc - w/2)
    y1 = max(0.0, yc - h/2)
    x2 = min(1.0, xc + w/2)
    y2 = max(0.0, yc - h/2)
    x3 = min(1.0, xc + w/2)
    y3 = min(1.0, yc + h/2)
    x4 = max(0.0, xc - w/2)
    y4 = min(1.0, yc + h/2)

    return f"{class_id} {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x3:.6f} {y3:.6f} {x4:.6f} {y4:.6f}\n"

def process_labels(dataset_path):
    fixed_count = 0
    # Process both train and val splits
    for split in ['train', 'val']:
        labels_dir = os.path.join(dataset_path, split, 'labels')
        if not os.path.exists(labels_dir):
            continue

        for filename in os.listdir(labels_dir):
            if not filename.endswith('.txt'): continue

            filepath = os.path.join(labels_dir, filename)
            lines = []
            modified = False

            with open(filepath, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    # If it has exactly 5 values, it's a bounding box
                    if len(parts) == 5:
                        lines.append(bbox_to_polygon(parts))
                        modified = True
                        fixed_count += 1
                    else:
                        lines.append(line)

            # Save the fixed labels back to the file
            if modified:
                with open(filepath, 'w') as f:
                    f.writelines(lines)

    return fixed_count

dataset_path = '/content/drive/MyDrive/SIH_Epilgoue/Dataset/Merged_YOLO_Dataset_v2'
print(f"Fixing labels in {dataset_path}...")
fixed = process_labels(dataset_path)
print(f"Fixed {fixed} bounding box annotations to polygons.")


In [ ]:
# 3. Train the Model
from ultralytics import YOLO
import torch

# Load a pre-trained YOLO segmentation model (Nano size)
model = YOLO('yolo11n-seg.pt')

# Define the path to your new merged data.yaml
data_path = '/content/drive/MyDrive/SIH_Epilgoue/Dataset/Merged_YOLO_Dataset_v2/data.yaml'

try:
    # Start Training!
    results = model.train(
        data=data_path,
        epochs=30,            # Adjust epochs as needed
        imgsz=640,            # Image size
        batch=16,             # Batch size (reduce to 8 if you hit out-of-memory errors)
        device=0 if torch.cuda.is_available() else 'cpu', # Use GPU if available
        patience=10,          # early stopping
        hsv_h=0.015,          # mild augmentations
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,          # Avoid destroying road geometry
        translate=0.1,
        scale=0.5,
        flipud=0.0,
        fliplr=0.5,
        mosaic=0.1,           # kept low for road scenes
        project='/content/drive/MyDrive/SIH_Epilgoue/Model/runs_small_v3',
        name='merged_idd_small_v3'
    )
except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("CUDA OOM error detected! Advice: Reduce batch to 8 and imgsz to 512 and re-run.")
    else:
        raise e


In [ ]:
# 4. Validate the Model
# Run evaluation metrics on the validation split
metrics = model.val()

In [ ]:
# 5. Test on an Image and Visualize Output
import cv2
import matplotlib.pyplot as plt
import glob

# 1. Load the trained model (pointing to the best weights from your training run)
model_path = "/content/drive/MyDrive/SIH_Epilgoue/Model/runs_small_v3/merged_idd_small_v3/weights/best.pt"
best_model = YOLO(model_path)

# 2. Pick an image to test (we will grab one from your validation set as an example)
test_images = glob.glob("/content/drive/MyDrive/SIH_Epilgoue/Dataset/Merged_YOLO_Dataset_v2/images/val/*.png") + \
              glob.glob("/content/drive/MyDrive/SIH_Epilgoue/Dataset/Merged_YOLO_Dataset_v2/images/val/*.jpg")

if len(test_images) > 0:
    image_path = test_images[0] # Test the first image we find
    print(f"Testing on image: {image_path}")

    # 3. Run prediction
    results = best_model.predict(image_path, conf=0.25)

    # 4. Plot and display the result
    res_plotted = results[0].plot() # This draws bounding boxes and masks on the image

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()
else:
    print("No validation images found to test.")
